# 01 Cleaning: from raw positions to one clean dataset

Raw files (one per time control) --> remove unusable rows --> mask suspicious clocks --> drop rows without a label --> keep and report missing engine values --> save `cleaned_all.csv` --> cleaning summary

This notebook replaces `Python Scripts/CleanData.py` and gives the same result. Every row is one position (one move) of a Lichess game.

**Cleaning rules**
- `y` (blunder yes/no) is never filled in. Rows without `y` are dropped (80 `inconsistent_mate_evaluations` rows).
- Missing `pre_eval_cp`, `played_eval_cp` and mate values stay missing. We add the column `pre_eval_cp_missing` (1 = missing).
- Negative `eval_loss_cp` and very large evaluations stay in the data. We only report them.
- Missing clocks are not filled in. Suspicious clocks are set to missing.
- No imputation and no scaling here. This happens later inside the cross-validation folds (`pipeline_models.ipynb`), so no information leaks from validation data into training.

In [65]:
#import
import os

import numpy as np
import pandas as pd

## 1. Data import

We read the three raw files (1+0, 5+0 and 10+0 games) and stack them into one dataset `master`. The column `source_pgn` tells us which file a row comes from.

In [66]:
#one raw file per time control
files = ["games_1m.csv", "games_5m.csv", "games_10m.csv"]

#read the three files and stack them into master
parts = []
for file in files:
    parts.append(pd.read_csv("../data/600000_Databasis/" + file, dtype={"game_id": str}))
master = pd.concat(parts, ignore_index=True)

print(f"{master['game_id'].nunique()} games, {len(master)} positions, {master.shape[1]} columns")

9000 games, 586827 positions, 86 columns


In [67]:
#games and positions per file
master.groupby("source_pgn", sort=False).agg(games=("game_id", "nunique"), positions=("ply", "size"))

,games,positions
source_pgn,,
games_1m.pgn,3000,194221
games_5m.pgn,3000,202576
games_10m.pgn,3000,190030


## 2. Remove unusable rows

A row is unusable if the game could not be read (`pgn_error`), if it has no game id, ply or player, or if the same position (`game_id`, `ply`) appears twice. In our data there are none of these rows, but the checks stay so that the notebook also works on new data.

The file also has an empty separator column `" "` between the features and the metadata. We drop it.

In [68]:
#rows where the pgn could not be read
pgn_error = master["label_status"] == "pgn_error"
master = master.loc[~pgn_error]

#rows without game id, ply or a valid player
invalid = master["game_id"].isna() | master["ply"].isna() | ~master["player_to_move"].isin(["white", "black"])
master = master.loc[~invalid]

#the same position twice: keep the first one
duplicated = master.duplicated(subset=["game_id", "ply"], keep="first")
master = master.loc[~duplicated]

#empty separator column
master = master.drop(columns=" ")

#infinite values count as missing
master = master.replace([np.inf, -np.inf], np.nan)

print(f"removed {pgn_error.sum()} pgn errors, {invalid.sum()} invalid rows, {duplicated.sum()} duplicates")

removed 0 pgn errors, 0 invalid rows, 0 duplicates


In [69]:
#each game must be one block of rows in move order (the clock checks below rely on this)
assert (master["game_id"] != master["game_id"].shift()).sum() == master["game_id"].nunique()
assert (master.groupby("game_id")["ply"].diff() > 0).sum() == len(master) - master["game_id"].nunique()

## 3. Mask suspicious clocks

All games are played without increment (1+0, 5+0, 10+0). A player's clock can therefore only go down during a game. A clock is **suspicious** if
- it is negative,
- it is higher than the starting time `initial_seconds` (or the starting time is missing),
- it is higher than the same player's previous known clock in this game, or
- the time spent on the move (`last_move_seconds`) is negative.

From the first suspicious move on, we set this player's clock to missing (NaN) for the rest of the game. We check white and black separately, and every check only uses moves up to the current one, so no information from later moves is used.

We do not fill the masked or missing clocks. The pipeline fills them later with the median of the training folds.

In [70]:
#the file stores the clock of the player to move and of the opponent; we rebuild white's and black's clock
white_to_move = master["player_to_move"] == "white"
white_clock = master["time_left_seconds"].where(white_to_move, master["opponent_time_left_seconds"])
black_clock = master["opponent_time_left_seconds"].where(white_to_move, master["time_left_seconds"])
clocks = {"white": white_clock, "black": black_clock}

games = master["game_id"]
start = master["initial_seconds"]

#one column per reason, per player
reasons = pd.DataFrame(index=master.index)
for color, clock in clocks.items():
    #last known clock of the same player before this move
    previous = clock.groupby(games).ffill().groupby(games).shift(1)

    reasons[color + "_negative_clock"] = clock < 0
    reasons[color + "_above_start"] = clock > start
    reasons[color + "_no_start_time"] = clock.notna() & start.isna()
    reasons[color + "_clock_went_up"] = clock > previous
    reasons[color + "_negative_move_time"] = (master["player_to_move"] == color) & (master["last_move_seconds"] < 0)

reasons.sum()

white_negative_clock          0
white_above_start           110
white_no_start_time           0
white_clock_went_up          79
white_negative_move_time     78
black_negative_clock          0
black_above_start            80
black_no_start_time           0
black_clock_went_up          79
black_negative_move_time     78
dtype: int64

In [71]:
#a player is suspicious from the first flagged move until the end of the game
suspicious = {}
for color in clocks:
    flagged = reasons.filter(like=color).any(axis=1)
    suspicious[color] = flagged.groupby(games).cummax()

#translate back to player to move and opponent
mover_suspicious = suspicious["white"].where(white_to_move, suspicious["black"])
opponent_suspicious = suspicious["black"].where(white_to_move, suspicious["white"])

#remember the raw clock values to count the masked rows later
clock_columns = ["time_left_seconds", "opponent_time_left_seconds", "last_move_seconds"]
raw_clocks = master[clock_columns].copy()

master.loc[mover_suspicious, "time_left_seconds"] = np.nan
master.loc[mover_suspicious, "last_move_seconds"] = np.nan
master.loc[opponent_suspicious, "opponent_time_left_seconds"] = np.nan

#a row counts as masked if at least one clock value was removed
clock_masked = (raw_clocks.notna() & master[clock_columns].isna()).any(axis=1)
print(f"{clock_masked.sum()} positions with masked clocks")

3962 positions with masked clocks


The other clock features are calculated from `time_left_seconds`, so we calculate them again with the masked values. Masked clocks give missing features, not wrong ones.

In [72]:
#recalculate the clock features from the masked clocks
seconds = master["time_left_seconds"]
master["time_left_fraction"] = (seconds / start).where(start > 0)
master["clock_difference"] = seconds - master["opponent_time_left_seconds"]
master["time_left_log1p"] = np.log1p(seconds)
master["low_time_10s"] = (seconds <= 10).astype(int).where(seconds.notna())

## 4. Drop rows without a label

`y = 1` if the move loses more than 200 centipawns, plus the mate rules (missed forced win, allowed forced loss). For 80 positions Stockfish gave inconsistent mate evaluations, so there is no `y`. We drop these rows and never fill in `y`.

We drop them only now, because the clock checks in section 3 also need these moves to know the previous clock.

In [73]:
#which rows have no label
print(master.loc[master["y"].isna(), "label_status"].value_counts())

master = master.dropna(subset=["y"])
master["y"] = master["y"].astype(int)

label_status
inconsistent_mate_evaluations    80
Name: count, dtype: int64


## 5. Missing engine values

`pre_eval_cp` (evaluation before the move, in centipawns) is missing when Stockfish sees a forced mate: then there is a mate distance (`pre_mate`) instead of a centipawn value. The same holds for `played_eval_cp` and `played_mate`. These rows have a valid `y`, so we keep them.

We do not fill in `pre_eval_cp`. We only add the indicator `pre_eval_cp_missing`, so the model can see that this position was a mate position.

In [74]:
#1 = pre_eval_cp is missing
master["pre_eval_cp_missing"] = master["pre_eval_cp"].isna().astype(int)

master[["pre_eval_cp", "played_eval_cp", "pre_mate", "played_mate"]].isna().sum()

pre_eval_cp        25862
played_eval_cp     29000
pre_mate          560885
played_mate       557747
dtype: int64

## 6. Negative losses and large evaluations: only report

`eval_loss_cp` can be negative: Stockfish searches with a fixed number of nodes, so the played move can look slightly better than the "best" move. Large evaluations (for example +2000) are real, clearly won positions. Both are no data errors, so we keep them and only report them.

In [75]:
#negative eval loss and very large evaluations (kept)
negative_loss = master["eval_loss_cp"] < 0
large_eval = master["pre_eval_cp"].abs() > 1000
print(f"{negative_loss.sum()} positions with negative eval loss, smallest {master['eval_loss_cp'].min():.0f} cp")
print(f"{large_eval.sum()} positions with |pre_eval_cp| > 1000 cp")

master["eval_loss_cp"].describe()

39018 positions with negative eval loss, smallest -1354 cp
18443 positions with |pre_eval_cp| > 1000 cp


count    553966.000000
mean         70.426228
std         158.159192
min       -1354.000000
25%           0.000000
50%           9.000000
75%          71.000000
max        7201.000000
Name: eval_loss_cp, dtype: float64

## 7. Save the clean dataset

Last checks, then we save one file. `pipeline_models.ipynb` reads exactly this file.

In [76]:
#sanity checks
assert not master.duplicated(subset=["game_id", "ply"]).any()
assert set(master["y"].unique()) == {0, 1}
assert (master["pre_eval_cp_missing"] == master["pre_eval_cp"].isna()).all()

os.makedirs("../data/processed", exist_ok=True)


#metadata has to be dropped  for the models (game_id and source_pgn are used for the split)
metadata = ["ply", "fen", "white_elo", "black_elo", "player_to_move", "utc_date", "utc_time",
            "played_move", "engine_best_move", "before_eval_depth", "played_eval_depth",
            "before_eval_nodes", "played_eval_nodes", "best_eval_status", "played_eval_status",
            "played_eval_cp", "pre_mate", "played_mate", "eval_loss_cp", "label_status", "pgn_error"]

master.drop(columns=metadata).to_csv("../data/processed/cleaned_all.csv", index=False)

print(f"{master['game_id'].nunique()} games, {len(master)} positions, blunder rate {master['y'].mean():.3f}")

9000 games, 586747 positions, blunder rate 0.115


## 8. Cleaning summary

In [77]:
#missing values per key column
key_columns = ["y", "pre_eval_cp", "played_eval_cp", "pre_mate", "played_mate",
               "time_left_seconds", "opponent_time_left_seconds", "last_move_seconds"]
missing = pd.DataFrame({"missing": master[key_columns].isna().sum(),
                        "share": master[key_columns].isna().mean().round(3)})
missing

,missing,share
y,0,0.000
pre_eval_cp,25862,0.044
played_eval_cp,29000,0.049
pre_mate,560885,0.956
played_mate,557747,0.951
time_left_seconds,21044,0.036
opponent_time_left_seconds,12122,0.021
last_move_seconds,39003,0.066


In [78]:
#masked clocks by reason (kept positions; one position can have several reasons)
kept_reasons = reasons.loc[master.index]
reason_names = ["negative_clock", "above_start", "no_start_time", "clock_went_up", "negative_move_time"]
masked_by_reason = pd.DataFrame({
    "white": [kept_reasons["white_" + r].sum() for r in reason_names],
    "black": [kept_reasons["black_" + r].sum() for r in reason_names]}, index=reason_names)
masked_by_reason["total"] = masked_by_reason["white"] + masked_by_reason["black"]
print(f"{clock_masked.loc[master.index].sum()} kept positions with masked clocks")
masked_by_reason

3962 kept positions with masked clocks


,white,black,total
negative_clock,0,0,0
above_start,110,80,190
no_start_time,0,0,0
clock_went_up,79,79,158
negative_move_time,78,78,156


In [79]:
#blunder rate and other counts per source file
master["clock_masked"] = clock_masked.loc[master.index]
summary = master.groupby("source_pgn", sort=False).agg(
    games=("game_id", "nunique"),
    positions=("y", "size"),
    blunder_rate=("y", "mean"),
    missing_pre_eval=("pre_eval_cp_missing", "sum"),
    negative_loss=("eval_loss_cp", lambda s: (s < 0).sum()),
    clock_masked=("clock_masked", "sum"))
master = master.drop(columns="clock_masked")
summary.round(3)

,games,positions,blunder_rate,missing_pre_eval,negative_loss,clock_masked
source_pgn,,,,,,
games_1m.pgn,3000,194191,0.119,8580,13026,617
games_5m.pgn,3000,202549,0.111,8565,13506,2051
games_10m.pgn,3000,190007,0.115,8717,12486,1294
